# 🔍 Nombres → DNI Scraper — Google Colab
Busca DNIs a partir de **Nombres + Apellido Paterno + Apellido Materno** usando `buscardniperu.com`

| Paso | Celda | Descripción |
|------|-------|-------------|
| 1️⃣ | **Instalación** | Instala dependencias *(solo una vez por sesión)* |
| 2️⃣ | **Código** | Carga el scraper en memoria |
| 3️⃣ | **Ejecutar** | Sube tu Excel → procesa → descarga resultado |

> ⚠️ Tu Excel debe tener las columnas: **`NOMBRES`**, **`APELLIDO_PATERNO`**, **`APELLIDO_MATERNO`**

## ⚙️ Celda 1 — Instalación *(solo una vez por sesión)*

In [ ]:
import subprocess, os, time

print('🌐 Instalando Google Chrome...')
subprocess.run(['wget', '-q', '-O', '/tmp/chrome.deb',
                'https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', '/tmp/chrome.deb'], check=True)

print('📦 Instalando librerías Python...')
subprocess.run(['pip', 'install', 'selenium', 'webdriver-manager',
                'pandas', 'openpyxl', 'ipywidgets', '--quiet'], check=True)

ver = subprocess.run(['google-chrome', '--version'], capture_output=True, text=True).stdout.strip()
print(f'\n✅ {ver}')
print('🎉 ¡Listo! Ejecuta la Celda 2 y luego la Celda 3.')

🌐 Instalando Google Chrome...
📦 Instalando librerías Python...

✅ Google Chrome 145.0.7632.159
🎉 ¡Listo! Ejecuta la Celda 2 y luego la Celda 3.


## 🧠 Celda 2 — Cargar el scraper

In [ ]:
import os, re, time, random, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from google.colab import files
from IPython.display import display, HTML
import ipywidgets as widgets

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# ── CONFIGURACIÓN ────────────────────────────────────────────────
HEADLESS       = True   # True = más rápido; False = ver el navegador (requiere Xvfb)
TIMEOUT        = 25     # segundos esperando resultados
PAUSA_MIN      = 3.0    # pausa mínima entre consultas
PAUSA_MAX      = 6.0    # pausa máxima entre consultas
GUARDAR_CADA   = 5      # guardar progreso cada N consultas

URL = 'https://buscardniperu.com/buscar-dni-por-nombres/'
_chromedriver_path = None

# ── DRIVER ───────────────────────────────────────────────────────
def _make_driver():
    global _chromedriver_path
    opt = Options()
    if HEADLESS:
        opt.add_argument('--headless=new')
    opt.add_argument('--no-sandbox')
    opt.add_argument('--disable-dev-shm-usage')
    opt.add_argument('--disable-gpu')
    opt.add_argument('--window-size=1280,900')
    opt.add_argument('--disable-blink-features=AutomationControlled')
    opt.add_argument('--disable-notifications')
    opt.add_argument('--disable-infobars')
    opt.add_argument('--disable-extensions')
    opt.add_argument(
        '--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36'
    )
    opt.binary_location = '/usr/bin/google-chrome'
    if _chromedriver_path is None:
        print('🔧 Descargando chromedriver...')
        _chromedriver_path = ChromeDriverManager().install()
        print('   ✅ chromedriver listo')
    driver = webdriver.Chrome(service=Service(_chromedriver_path), options=opt)
    driver.set_page_load_timeout(30)
    return driver

# ── UTILIDADES JS ────────────────────────────────────────────────
def _js_limpiar_overlays(driver):
    driver.execute_script("""
        ['iframe','[id*="ads"]','[class*="ads"]','.fc-consent-root',
         '.fc-overlay','.modal','.popup','[role="dialog"]']
        .forEach(sel => document.querySelectorAll(sel).forEach(e => e.remove()));
        document.body.style.overflow = 'auto';
    """)

def _js_aceptar_cookies(driver):
    driver.execute_script("""
        const b = Array.from(document.querySelectorAll('button'))
            .find(x => ['acept','ok','entendido']
            .some(t => (x.innerText||'').toLowerCase().includes(t)));
        if (b) b.click();
    """)

def _js_click_buscar(driver):
    return driver.execute_script("""
        const btn = document.querySelector('button.btn-danger');
        if (!btn) return false;
        btn.scrollIntoView({block:'center'});
        btn.click();
        return true;
    """)

def _construir_clave(nombres, ap_pat, ap_mat):
    def norm(x):
        x = str(x).strip().upper()
        return re.sub(r'\s+', ' ', x)
    return f'{norm(nombres)}|{norm(ap_pat)}|{norm(ap_mat)}'

# ── SCRAPER INDIVIDUAL ───────────────────────────────────────────
def buscar_dnis(driver, nombres, ap_pat, ap_mat):
    """
    Retorna (status, resultados, error_msg)
    status: 'OK' | 'SIN_RESULTADOS' | 'ERROR'
    resultados: [{'dni':..., 'nombre_completo':...}]
    """
    try:
        driver.get(URL)
        time.sleep(2)
        for _ in range(3):
            _js_limpiar_overlays(driver)
            _js_aceptar_cookies(driver)
            time.sleep(0.5)

        wait = WebDriverWait(driver, TIMEOUT)
        wait.until(EC.presence_of_element_located((By.NAME, 'ap_pat')))

        for campo_name, valor in [('ap_pat', ap_pat), ('ap_mat', ap_mat), ('nombres', nombres)]:
            campo = driver.find_element(By.NAME, campo_name)
            campo.clear()
            campo.send_keys(str(valor))

        time.sleep(0.8)
        _js_limpiar_overlays(driver)

        if not _js_click_buscar(driver):
            return 'ERROR', [], 'No se encontró botón Buscar'

        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, '.resultados table tbody')))
        time.sleep(1)

        filas = driver.find_elements(By.CSS_SELECTOR, '.resultados table tbody tr')
        resultados = []
        for f in filas:
            celdas = f.find_elements(By.TAG_NAME, 'td')
            if len(celdas) >= 4:
                dni = celdas[0].text.strip()
                nombre = f"{celdas[1].text} {celdas[2].text} {celdas[3].text}".strip()
                if dni.isdigit():
                    resultados.append({'dni': dni, 'nombre_completo': nombre})

        if not resultados:
            return 'SIN_RESULTADOS', [], ''
        return 'OK', resultados, ''

    except Exception as e:
        return 'ERROR', [], str(e)

# ── BUILD WIDE ───────────────────────────────────────────────────
def construir_wide(df_madre, df_long):
    if df_long.empty:
        return df_madre.copy()
    df_long = df_long.copy()
    df_long['rank'] = df_long['rank'].astype(int)
    df_long = df_long.sort_values(['clave_busqueda', 'rank'])

    dni_wide = df_long.pivot(index='clave_busqueda', columns='rank', values='dni')
    dni_wide.columns = [f'DNI_{int(c)}' for c in dni_wide.columns]
    dni_wide = dni_wide.reset_index()

    nom_wide = df_long.pivot(index='clave_busqueda', columns='rank', values='nombre_completo_resultado')
    nom_wide.columns = [f'NOMBRE_COMPLETO_{int(c)}' for c in nom_wide.columns]
    nom_wide = nom_wide.reset_index()

    wide = dni_wide.merge(nom_wide, on='clave_busqueda', how='left')
    return df_madre.merge(wide, on='clave_busqueda', how='left')

# ── CLASE PRINCIPAL ──────────────────────────────────────────────
class NombresDNIScraper:

    def __init__(self, excel_path, output_path=None):
        self.excel_path  = excel_path
        base, ext        = os.path.splitext(excel_path)
        self.output_path = output_path or f'{base}_resultado{ext}'
        self._procesados = 0
        self._total      = 0
        self._inicio     = None

    def _actualizar_barra(self, barra, label, eta_label, clave, status, n_res):
        self._procesados += 1
        pct         = int(self._procesados / self._total * 100)
        barra.value = self._procesados

        # Tiempo estimado restante
        elapsed  = time.time() - self._inicio
        por_item = elapsed / self._procesados
        restante = por_item * (self._total - self._procesados)
        eta_str  = str(timedelta(seconds=int(restante)))
        vel_str  = f'{por_item:.1f}s/consulta'

        icono = '✅' if status == 'OK' else ('⚠️' if status == 'SIN_RESULTADOS' else '❌')
        nombres_corto = clave.split('|')[0][:20]

        label.value = (
            f'<b style="font-size:14px">{self._procesados}/{self._total} — {pct}%</b>'
            f' &nbsp;|&nbsp; {icono} {nombres_corto}: {n_res} resultado(s)'
        )
        eta_label.value = (
            f'<span style="color:#555;font-size:12px">'
            f'⏱ Tiempo restante estimado: <b>{eta_str}</b> &nbsp;|&nbsp; '
            f'Velocidad: <b>{vel_str}</b></span>'
        )

    def procesar_excel(self):
        print(f'📄 Leyendo: {self.excel_path}')
        df = pd.read_excel(self.excel_path, dtype=str)
        df.columns = [c.strip().upper() for c in df.columns]

        # Aceptar variantes de nombre de columna
        col_map = {}
        for col in df.columns:
            if 'NOMBRE' in col and 'COMPLETO' not in col and 'PATERNO' not in col and 'MATERNO' not in col:
                col_map.setdefault('NOMBRES', col)
            if 'PATERNO' in col:
                col_map['APELLIDO_PATERNO'] = col
            if 'MATERNO' in col:
                col_map['APELLIDO_MATERNO'] = col

        faltan = {'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO'} - set(col_map.keys())
        if faltan:
            raise ValueError(f'❌ Faltan columnas: {faltan}. Columnas encontradas: {list(df.columns)}')

        # Renombrar a nombres estándar
        df = df.rename(columns={v: k for k, v in col_map.items()})
        for c in ['NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO']:
            df[c] = df[c].fillna('').astype(str)

        # Construir claves
        df['clave_busqueda'] = df.apply(
            lambda r: _construir_clave(r['NOMBRES'], r['APELLIDO_PATERNO'], r['APELLIDO_MATERNO']), axis=1
        )

        claves = df['clave_busqueda'].drop_duplicates().tolist()
        total  = len(claves)
        self._total  = total
        self._inicio = time.time()

        # Estimación inicial de tiempo
        pausa_promedio = (PAUSA_MIN + PAUSA_MAX) / 2 + 3  # pausa + tiempo carga
        segundos_est   = int(total * pausa_promedio)
        eta_inicial    = str(timedelta(seconds=segundos_est))

        print(f'📊 Personas únicas a consultar: {total}')
        print(f'⏱  Tiempo estimado: ~{eta_inicial} (a ~{pausa_promedio:.0f}s por consulta)')
        print()

        # Widgets
        barra = widgets.IntProgress(
            value=0, min=0, max=total, description='Progreso:',
            bar_style='info', layout=widgets.Layout(width='100%')
        )
        label     = widgets.HTML(value=f'<b style="font-size:14px">0/{total} — 0%</b>')
        eta_label = widgets.HTML(
            value=f'<span style="color:#555;font-size:12px">'
                  f'⏱ Tiempo estimado total: <b>~{eta_inicial}</b></span>'
        )
        display(widgets.VBox([barra, label, eta_label]))

        df_madre = pd.DataFrame(columns=[
            'clave_busqueda','nombres','apellido_paterno','apellido_materno',
            'status','n_resultados','fecha_consulta','error_msg'
        ])
        df_long = pd.DataFrame(columns=[
            'clave_busqueda','rank','dni','nombre_completo_resultado'
        ])

        driver  = _make_driver()
        nuevas  = 0

        try:
            for clave in claves:
                fila    = df.loc[df['clave_busqueda'] == clave].iloc[0]
                nombres = fila['NOMBRES']
                ap_pat  = fila['APELLIDO_PATERNO']
                ap_mat  = fila['APELLIDO_MATERNO']

                status, resultados, error_msg = buscar_dnis(driver, nombres, ap_pat, ap_mat)
                fecha  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                n_res  = len(resultados)

                df_madre = pd.concat([df_madre, pd.DataFrame([{
                    'clave_busqueda': clave, 'nombres': nombres,
                    'apellido_paterno': ap_pat, 'apellido_materno': ap_mat,
                    'status': status, 'n_resultados': str(n_res),
                    'fecha_consulta': fecha, 'error_msg': error_msg or ''
                }])], ignore_index=True)

                if status == 'OK' and n_res > 0:
                    filas_long = [{
                        'clave_busqueda': clave, 'rank': str(i),
                        'dni': r['dni'], 'nombre_completo_resultado': r['nombre_completo']
                    } for i, r in enumerate(resultados, 1)]
                    df_long = pd.concat([df_long, pd.DataFrame(filas_long)], ignore_index=True)

                nuevas += 1
                self._actualizar_barra(barra, label, eta_label, clave, status, n_res)

                if nuevas % GUARDAR_CADA == 0:
                    self._guardar(df, df_madre, df_long)

                time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

        finally:
            try:
                driver.quit()
            except Exception:
                pass

        self._guardar(df, df_madre, df_long)

        # Fin
        barra.bar_style = 'success'
        barra.value     = total
        elapsed_total   = str(timedelta(seconds=int(time.time() - self._inicio)))
        ok_count        = (df_madre['status'] == 'OK').sum()
        sin_count       = (df_madre['status'] == 'SIN_RESULTADOS').sum()
        err_count       = (df_madre['status'] == 'ERROR').sum()

        label.value = f'<b style="font-size:14px;color:green">✅ {total}/{total} — 100% COMPLETADO</b>'
        eta_label.value = (
            f'<span style="color:#333;font-size:12px">'
            f'⏱ Tiempo total real: <b>{elapsed_total}</b></span>'
        )
        print(f'\n✅ ¡Listo!')
        print(f'   📁 Guardado en    : {self.output_path}')
        print(f'   ✅ Con resultados : {ok_count}')
        print(f'   ⚠️  Sin resultados : {sin_count}')
        print(f'   ❌ Errores        : {err_count}')
        print(f'   ⏱  Tiempo total   : {elapsed_total}')

    def _guardar(self, df_input, df_madre, df_long):
        df_wide = construir_wide(df_madre, df_long)
        with pd.ExcelWriter(self.output_path, engine='openpyxl') as writer:
            df_wide.to_excel(writer, sheet_name='Resultados', index=False)
            df_long.to_excel(writer, sheet_name='Detalle_long', index=False)
            df_madre.to_excel(writer, sheet_name='Log', index=False)

print('✅ Scraper listo. Ejecuta la Celda 3.')

✅ Scraper listo. Ejecuta la Celda 3.


## 🚀 Celda 3 — Subir Excel y ejecutar

In [ ]:
display(HTML('<b>📂 Selecciona tu Excel (columnas: NOMBRES, APELLIDO_PATERNO, APELLIDO_MATERNO):</b>'))
uploaded = files.upload()

if not uploaded:
    print('❌ No se subió ningún archivo.')
else:
    for nombre_original, contenido in uploaded.items():
        nombre_limpio = re.sub(r' \(\d+\)(\.[^.]+)$', r'\1', nombre_original)
        with open(nombre_limpio, 'wb') as f:
            f.write(contenido)

        base, ext     = os.path.splitext(nombre_limpio)
        nombre_salida = f'{base}_resultado{ext}'

        print(f'📌 Entrada : {nombre_limpio}')
        print(f'📌 Salida  : {nombre_salida}\n')

        scraper = NombresDNIScraper(excel_path=nombre_limpio, output_path=nombre_salida)
        scraper.procesar_excel()

        print('\n⬇️  Descargando resultado...')
        files.download(nombre_salida)

Saving PRUEBA_NOMBRES_DNI.xlsx to PRUEBA_NOMBRES_DNI.xlsx
📌 Entrada : PRUEBA_NOMBRES_DNI.xlsx
📌 Salida  : PRUEBA_NOMBRES_DNI_resultado.xlsx

📄 Leyendo: PRUEBA_NOMBRES_DNI.xlsx
📊 Personas únicas a consultar: 10
⏱  Tiempo estimado: ~0:01:15 (a ~8s por consulta)



🔧 Descargando chromedriver...
   ✅ chromedriver listo

✅ ¡Listo!
   📁 Guardado en    : PRUEBA_NOMBRES_DNI_resultado.xlsx
   ✅ Con resultados : 9
   ⚠️  Sin resultados : 1
   ❌ Errores        : 0
   ⏱  Tiempo total   : 0:02:02

⬇️  Descargando resultado...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>